In [ ]:
# ==============================
# 1. IMPORTS & LOAD DATA
# ==============================

# Basic data manipulation and math
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Skewness calculation
from scipy.stats import skew

# Encoding categorical variables
from sklearn.preprocessing import LabelEncoder

# Load the training and test datasets from Kaggle's competition input directory
train = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/train.csv")
test = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/test.csv")

# Check the size of datasets and preview first rows
print("Train shape:", train.shape)
print("Test shape:", test.shape)
train.head()

In [ ]:
# ==============================
# 2. EXPLORATORY DATA ANALYSIS (VISUALIZATIONS)
# ==============================

sns.set(style="whitegrid", palette="muted", font_scale=1.1)

# 1. Distribution of SalePrice
plt.figure(figsize=(8,6))
sns.histplot(train["SalePrice"], kde=True, bins=40)
plt.title("Distribution of SalePrice")
plt.show()

# 2. SalePrice vs Overall Quality (boxplot shows price spread per quality rating)
plt.figure(figsize=(8,6))
sns.boxplot(x="OverallQual", y="SalePrice", data=train)
plt.title("SalePrice vs Overall Quality")
plt.show()

# 3. Scatterplot: SalePrice vs Living Area (check linear relationship/outliers)
plt.figure(figsize=(8,6))
sns.scatterplot(x="GrLivArea", y="SalePrice", data=train)
plt.title("SalePrice vs Living Area (GrLivArea)")
plt.show()

# 4. Correlation with SalePrice (Top 10 features)
corr = train.corr(numeric_only=True)["SalePrice"].sort_values(ascending=False).head(10)
plt.figure(figsize=(8,6))
sns.barplot(x=corr.values, y=corr.index)
plt.title("Top 10 Correlations with SalePrice")
plt.show()

# 5. Missing values overview (Top 20 features with most missing data)
missing = train.isnull().mean().sort_values(ascending=False)[:20]
plt.figure(figsize=(8,6))
sns.barplot(x=missing.values, y=missing.index)
plt.title("Top 20 Features with Missing Values")
plt.show()

In [ ]:
# ==============================
# 3. DATA CLEANING (FIXED)
# ==============================

# Drop features with more than 40% missing values
threshold = 0.4
drop_cols = train.columns[train.isnull().mean() > threshold]
train = train.drop(columns=drop_cols)
test = test.drop(columns=drop_cols, errors="ignore")  # ignore missing columns in test

# Identify numeric features, excluding target "SalePrice"
num_cols = train.select_dtypes(include=["int64", "float64"]).columns.drop("SalePrice")

# Fill numeric missing values with median
for col in num_cols:
    train[col] = train[col].fillna(train[col].median())
    if col in test.columns:  # only fill if column exists in test
        test[col] = test[col].fillna(train[col].median())

# Fill categorical missing values with mode
cat_cols = train.select_dtypes(include=["object"]).columns
for col in cat_cols:
    train[col] = train[col].fillna(train[col].mode()[0])
    if col in test.columns:  # same fix for test set
        test[col] = test[col].fillna(test[col].mode()[0])

In [ ]:
# ==============================
# 4. FEATURE ENGINEERING WITH PCA
# ==============================

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.decomposition import PCA

# ---- Target Transformation ----
# Apply log1p to SalePrice to reduce skew
y = np.log1p(train["SalePrice"])  # target variable
train_features = train.drop("SalePrice", axis=1)

# ---- One-Hot Encoding for categoricals ----
cat_cols = train_features.select_dtypes(include=["object"]).columns
num_cols = train_features.select_dtypes(include=["int64", "float64"]).columns

# OneHotEncode categorical variables
encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)
encoded_train = encoder.fit_transform(train_features[cat_cols])
encoded_test = encoder.transform(test[cat_cols])

# Create encoded DataFrame with proper column names
encoded_train_df = pd.DataFrame(encoded_train, 
                                columns=encoder.get_feature_names_out(cat_cols),
                                index=train.index)
encoded_test_df = pd.DataFrame(encoded_test, 
                               columns=encoder.get_feature_names_out(cat_cols),
                               index=test.index)

# Reconstruct full feature set (numeric + encoded categorical)
train_ready = pd.concat([train_features[num_cols], encoded_train_df], axis=1)
test_ready = pd.concat([test[num_cols], encoded_test_df], axis=1)

# ---- Scale features before PCA ----
scaler = StandardScaler()
X_scaled = scaler.fit_transform(train_ready)
X_test_scaled = scaler.transform(test_ready)

# ---- Apply PCA ----
# Keep enough components to explain ~95% variance
pca = PCA(n_components=0.95, random_state=42)
X_pca = pca.fit_transform(X_scaled)
X_test_pca = pca.transform(X_test_scaled)

print("Original feature count:", train_ready.shape[1])
print("Reduced feature count (PCA):", X_pca.shape[1])

In [ ]:
# ==============================
# LINEAR REGRESSION
# ==============================
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score
import numpy as np

lin_reg = LinearRegression()
lin_reg_scores = np.sqrt(-cross_val_score(
    lin_reg, X_pca, y,
    scoring="neg_mean_squared_error", cv=5
))

print("Linear Regression RMSE:", lin_reg_scores.mean())

In [ ]:
# ==============================
# NEURAL NETWORK (MLP)
# ==============================
from sklearn.neural_network import MLPRegressor

mlp = MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42)
mlp_scores = np.sqrt(-cross_val_score(
    mlp, X_pca, y,
    scoring="neg_mean_squared_error", cv=5
))

print("Neural Net (MLP) RMSE:", mlp_scores.mean())

In [ ]:
# ==============================
# DECISION TREE
# ==============================
from sklearn.tree import DecisionTreeRegressor

tree = DecisionTreeRegressor(random_state=42)
tree_scores = np.sqrt(-cross_val_score(
    tree, X_pca, y,
    scoring="neg_mean_squared_error", cv=5
))

print("Decision Tree RMSE:", tree_scores.mean())

In [ ]:
# ==============================
# RANDOM FOREST
# ==============================
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=10, random_state=42)
rf_scores = np.sqrt(-cross_val_score(
    rf, X_pca, y,
    scoring="neg_mean_squared_error", cv=5
))

print("Random Forest RMSE:", rf_scores.mean())

In [ ]:
# ==============================
# GRADIENT BOOSTING
# ==============================
from sklearn.ensemble import GradientBoostingRegressor

gbr = GradientBoostingRegressor(n_estimators=10, random_state=42)
gbr_scores = np.sqrt(-cross_val_score(
    gbr, X_pca, y,
    scoring="neg_mean_squared_error", cv=5
))

print("Gradient Boosting RMSE:", gbr_scores.mean())

In [ ]:
# ==============================
# XGBOOST
# ==============================
import xgboost as xgb

xgb_model = xgb.XGBRegressor(n_estimators=10, learning_rate=0.05, random_state=42)
xgb_scores = np.sqrt(-cross_val_score(
    xgb_model, X_pca, y,
    scoring="neg_mean_squared_error", cv=5
))

print("XGBoost RMSE:", xgb_scores.mean())

In [ ]:
# ==============================
# LIGHTGBM
# ==============================
import lightgbm as lgb

lgb_model = lgb.LGBMRegressor(n_estimators=10, learning_rate=0.05, random_state=42)
lgb_scores = np.sqrt(-cross_val_score(
    lgb_model, X_pca, y,
    scoring="neg_mean_squared_error", cv=5
))

print("LightGBM RMSE:", lgb_scores.mean())

In [ ]:
# ==============================
# CATBOOST
# ==============================
import catboost as cb

cat_model = cb.CatBoostRegressor(iterations=50, learning_rate=0.05,
                                 depth=6, verbose=0, random_state=42)
cat_scores = np.sqrt(-cross_val_score(
    cat_model, X_pca, y,
    scoring="neg_mean_squared_error", cv=5
))

print("CatBoost RMSE:", cat_scores.mean())

In [ ]:
# ==============================
# MODEL COMPARISON TABLE
# ==============================

import pandas as pd

# Collect all scores into a dictionary
results = {
    "Linear Regression": lin_reg_scores.mean(),
    "Neural Net (MLP)": mlp_scores.mean(),
    "Decision Tree": tree_scores.mean(),
    "Random Forest": rf_scores.mean(),
    "Gradient Boosting": gbr_scores.mean(),
    "XGBoost": xgb_scores.mean(),
    "LightGBM": lgb_scores.mean(),
    "CatBoost": cat_scores.mean()
}

# Convert to DataFrame
results_df = pd.DataFrame(list(results.items()), columns=["Model", "RMSE"]).sort_values(by="RMSE")
display(results_df)

# Identify the best model
best_model_name = results_df.iloc[0]["Model"]
print("Best model based on validation RMSE:", best_model_name)

In [ ]:
# ==============================
# FEATURE IMPORTANCE (TOP 10)
# ==============================

import matplotlib.pyplot as plt
import seaborn as sns

# Refit the best model on the full training set
best_model = None
if best_model_name == "Linear Regression":
    best_model = lin_reg.fit(X_pca, y)
elif best_model_name == "Neural Net (MLP)":
    best_model = mlp.fit(X_pca, y)
elif best_model_name == "Decision Tree":
    best_model = tree.fit(X_pca, y)
elif best_model_name == "Random Forest":
    best_model = rf.fit(X_pca, y)
elif best_model_name == "Gradient Boosting":
    best_model = gbr.fit(X_pca, y)
elif best_model_name == "XGBoost":
    best_model = xgb_model.fit(X_pca, y)
elif best_model_name == "LightGBM":
    best_model = lgb_model.fit(X_pca, y)
elif best_model_name == "CatBoost":
    best_model = cat_model.fit(X_pca, y)

# Plot feature importance if available
if hasattr(best_model, "feature_importances_"):
    importances = best_model.feature_importances_
    indices = np.argsort(importances)[-10:]  # Top 10
    top_features = [f"PC{i+1}" for i in indices]  # PCA components
    
    plt.figure(figsize=(8,6))
    sns.barplot(x=importances[indices], y=top_features, palette="viridis")
    plt.title(f"Top 10 Feature Importances - {best_model_name}")
    plt.xlabel("Importance")
    plt.ylabel("Principal Components")
    plt.show()
else:
    print(f"{best_model_name} does not provide feature importances.")

In [ ]:
# =====================================
# DATA VISUALIZATION & SUBMISSION
# =====================================

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor
import pandas as pd

# --------------------------
# 1️⃣ Data Visualization
# --------------------------

# --- 1. Distribution of SalePrice ---
plt.figure(figsize=(8,6))
sns.histplot(y, kde=True, bins=30, color="blue")
plt.title("Distribution of SalePrice")
plt.xlabel("SalePrice")
plt.ylabel("Frequency")
plt.show()

# --- 2. Feature correlation heatmap (top 15 numeric features) ---
numeric_cols = train.select_dtypes(include=["int64", "float64"]).columns
corr = train[numeric_cols].corr()["SalePrice"].sort_values(ascending=False)[:15]

plt.figure(figsize=(12,8))
sns.barplot(x=corr.values, y=corr.index, palette="viridis")
plt.title("Top 15 Numeric Features Correlated with SalePrice")
plt.xlabel("Correlation")
plt.show()

# --------------------------
# 2️⃣ Train/Validation Split for Visualization
# --------------------------

X_tr, X_val, y_tr, y_val = train_test_split(X_pca, y, test_size=0.2, random_state=42)

# --------------------------
# 3️⃣ Instantiate final model (for visualization)
# --------------------------

# Replace with your selected best model name
best_model_name = "LightGBM"  

if best_model_name == "Linear Regression":
    final_model = LinearRegression()
elif best_model_name == "Neural Net (MLP)":
    final_model = MLPRegressor(hidden_layer_sizes=(64,32), max_iter=50, random_state=42)
elif best_model_name == "Decision Tree":
    final_model = DecisionTreeRegressor(random_state=42)
elif best_model_name == "Random Forest":
    final_model = RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1)
elif best_model_name == "Gradient Boosting":
    final_model = GradientBoostingRegressor(n_estimators=50, learning_rate=0.05, random_state=42)
elif best_model_name == "XGBoost":
    final_model = xgb.XGBRegressor(n_estimators=50, learning_rate=0.05, random_state=42)
elif best_model_name == "LightGBM":
    final_model = lgb.LGBMRegressor(n_estimators=50, learning_rate=0.05, random_state=42, n_jobs=-1)
elif best_model_name == "CatBoost":
    final_model = CatBoostRegressor(iterations=50, learning_rate=0.05, verbose=0, random_state=42)

# Fit on training split
final_model.fit(X_tr, y_tr)
y_val_pred = final_model.predict(X_val)

# --- 3. Residuals plot ---
residuals = y_val - y_val_pred
plt.figure(figsize=(8,6))
sns.scatterplot(x=y_val_pred, y=residuals, alpha=0.6)
plt.axhline(0, color="red", linestyle="--")
plt.title(f"Residuals Plot ({best_model_name})")
plt.xlabel("Predicted SalePrice")
plt.ylabel("Residuals (y - ŷ)")
plt.show()

# --- 4. Predicted vs Actual SalePrice ---
plt.figure(figsize=(8,6))
sns.scatterplot(x=y_val, y=y_val_pred, alpha=0.6)
plt.plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()], "r--")
plt.title(f"Predicted vs Actual SalePrice ({best_model_name})")
plt.xlabel("Actual SalePrice")
plt.ylabel("Predicted SalePrice")
plt.show()

# --- 5. Feature importance (Top 10 PCA components if available) ---
if hasattr(final_model, "feature_importances_"):
    importances = final_model.feature_importances_
    top_idx = np.argsort(importances)[-10:]
    
    plt.figure(figsize=(10,6))
    plt.barh(range(len(top_idx)), importances[top_idx], align="center")
    plt.yticks(range(len(top_idx)), [f"PC{i}" for i in top_idx])
    plt.xlabel("Importance")
    plt.title(f"Top 10 PCA Feature Importances ({best_model_name})")
    plt.show()
else:
    print(f"{best_model_name} does not provide feature importances.")

# --------------------------
# 6️⃣ Final Model Training on Full Data & Submission
# --------------------------

# Transform the full training and test set using PCA
X_full_pca = pca.fit_transform(X_scaled)
X_test_pca = pca.transform(X_test_scaled)

# Re-train final model on full training data
final_model.fit(X_full_pca, y)

# Make predictions on test set
y_pred = np.expm1(final_model.predict(X_test_pca))  # convert back from log scale if log-transformed

# Build submission DataFrame
submission = pd.DataFrame({
    "Id": test["Id"],  # ensure test DataFrame is named 'test'
    "SalePrice": y_pred
})

# Save submission CSV
submission.to_csv("submission.csv", index=False)
print("✅ Submission file created: submission.csv")